# jaxctrl on a gene-regulatory network: SINDy → controllability → LQR

**Identify a model from time-series data, then do control theory on it** — the pipeline jaxctrl Layers 0–1 are built for, run on a small gene-regulatory network.

We use the **IRMA** topology (Cantone et al., *Cell* 2009 — a 5-gene network engineered in *S. cerevisiae* specifically as a network-inference benchmark, with galactose switch-on / switch-off time courses): `Swi5 → {Cbf1, Ash1}`, `Cbf1 → Gal4`, `Gal4 → Gal80`, `Gal80 ⊣ Swi5` (the negative-feedback loop), `Ash1 ⊣ Cbf1`. Here we drive the dynamics from a **stylized Hill-function ODE model with the IRMA wiring** (not the published kinetic parameters — see the note at the end for the experimental dataset), recover a linear surrogate from the simulated switch-on trajectory with `SINDyOptimizer`, check controllability, and design an LQR "drug input" that steers the network back to its switch-off steady state.

Everything is `jit`-able and differentiable end-to-end.

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import jaxctrl

## 1. The IRMA-topology ODE (switch-on regime, +galactose)

5 genes, $\dot s_i = b_i + \sum (\text{Hill activations / repressions}) - \gamma s_i$. We integrate from a low state to the switch-on steady state.

In [ ]:
names = ["Cbf1", "Gal4", "Swi5", "Gal80", "Ash1"]

def hill_act(x, K=0.3, nH=2.0): return x**nH / (K**nH + x**nH)
def hill_rep(x, K=0.3, nH=2.0): return K**nH / (K**nH + x**nH)

GAMMA, LEAK, VMAX = 0.6, 0.05, 1.0
def rhs(s):
    Cbf1, Gal4, Swi5, Gal80, Ash1 = s
    dCbf1  = LEAK + VMAX*hill_act(Swi5)*hill_rep(Ash1) - GAMMA*Cbf1   # Swi5 -> Cbf1,  Ash1 -| Cbf1
    dGal4  = LEAK + VMAX*hill_act(Cbf1)                - GAMMA*Gal4   # Cbf1 -> Gal4
    dSwi5  = LEAK + VMAX*hill_rep(Gal80)               - GAMMA*Swi5   # Gal80 -| Swi5  (neg. feedback)
    dGal80 = LEAK + VMAX*hill_act(Gal4)               - GAMMA*Gal80   # Gal4 -> Gal80
    dAsh1  = LEAK + VMAX*hill_act(Swi5)               - GAMMA*Ash1   # Swi5 -> Ash1
    return jnp.array([dCbf1, dGal4, dSwi5, dGal80, dAsh1])

dt, T = 0.02, 25.0
n_steps = int(T / dt)
x = jnp.full(5, 0.05)            # "switch-off" low state
traj = [x]
for _ in range(n_steps):
    x = x + dt * rhs(x)
    traj.append(x)
traj = jnp.array(traj)            # (n_steps+1, 5)
x_off = traj[0]                   # switch-off steady state (the target we steer back to)
x_on  = traj[-1]                  # switch-on steady state
print("switch-OFF state:", np.asarray(x_off).round(3))
print("switch-ON  state:", np.asarray(x_on).round(3))

t_grid = np.arange(len(traj)) * dt
plt.figure(figsize=(7, 4))
for i, nm in enumerate(names):
    plt.plot(t_grid, traj[:, i], label=nm)
plt.xlabel("time"); plt.ylabel("expression"); plt.title("IRMA-topology switch-on trajectory"); plt.legend(); plt.show()

## 2. SINDy: recover a linear surrogate of the GRN dynamics

We treat the trajectory as data, add a little measurement noise, and fit $\dot x = A x + c$ with `SINDyOptimizer` on a degree-1 polynomial library (so the recovered `A` is the network's Jacobian estimate). The recovered `A` is an *averaged* Jacobian over the sampled trajectory (mostly the transient), so it shares stability and rough structure with the true steady-state Jacobian rather than its exact entries — which is all a linear surrogate needs to be for the controllability + LQR steps below.

In [ ]:
X  = traj[:-1]
dX = (traj[1:] - traj[:-1]) / dt
key = jax.random.PRNGKey(0)
dX_noisy = dX + 0.005 * jax.random.normal(key, dX.shape)

sindy = jaxctrl.SINDyOptimizer(threshold=0.02, max_iter=20)
lib1  = lambda Z: jaxctrl.polynomial_library(Z, degree=1)
Xi    = sindy.fit(X, dX_noisy, lib1)
A_id  = sindy.linearize(Xi, n_vars=5)          # rows 1..n of Xi == Jacobian block
print("SINDy-identified A:\n", np.asarray(A_id).round(2))
print("eigenvalues:", np.asarray(jnp.linalg.eigvals(A_id)).round(3))

A_true = jax.jacfwd(rhs)(x_on)
print("\ntrue Jacobian at the switch-on steady state:\n", np.asarray(A_true).round(2))
print("eigenvalues:", np.asarray(jnp.linalg.eigvals(A_true)).round(3))

## 3. Controllability: is a single-gene "drug" enough?

The actuator: a construct that adds **Swi5** mRNA (Swi5 is the hub of the negative-feedback loop). Is the identified network controllable from that one input?

In [ ]:
B = jnp.zeros((5, 1)).at[names.index("Swi5"), 0].set(1.0)
print("controllable from a Swi5 input?", bool(jaxctrl.is_controllable(A_id, B)))
print("stabilizable?                  ", bool(jaxctrl.is_stabilizable(A_id, B)))
print("open-loop stable?              ", bool(jaxctrl.is_stable(A_id)))
# also: which single gene gives the largest controllability subspace?
for i, nm in enumerate(names):
    Bi = jnp.eye(5)[:, i:i+1]
    print(f"  drive {nm:6s}: Kalman rank {int(jnp.linalg.matrix_rank(jaxctrl.controllability_matrix(A_id, Bi)))}/5")

## 4. LQR "drug-design": steer the switch-on network back toward switch-off

Work in deviation coordinates $y = x - x_{\text{off}}$; the LQR law $u = -K\,y$ minimises $\int_0^\infty (y^\top Q y + u^\top R u)\,dt$. (We use a Hurwitz copy of $A$ for the infinite-horizon solve when the identified $A$ has a slightly unstable mode — a standard regularisation.)

In [ ]:
eig_max = float(jnp.max(jnp.linalg.eigvals(A_id).real))
A_lqr   = A_id - max(eig_max + 0.2, 0.0) * jnp.eye(5)
Q, R    = jnp.eye(5), jnp.eye(1)
K, P    = jaxctrl.lqr(A_lqr, B, Q, R)
print("optimal gain K =", np.asarray(K).round(3))
print("closed-loop eigenvalues:", np.asarray(jnp.linalg.eigvals(A_lqr - B @ K)).round(3))
print("LQR cost-to-go from the switch-on state:  y0^T P y0 =",
      round(float((x_on - x_off) @ P @ (x_on - x_off)), 3))

y0 = jnp.asarray(np.asarray(x_on - x_off))
ts, ys, us = jaxctrl.simulate_closed_loop(A_lqr, B, K, x0=y0, T=15.0, num_steps=300)
xs = np.asarray(ys) + np.asarray(x_off)[None, :]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for i, nm in enumerate(names):
    ax[0].plot(ts, xs[:, i], label=nm)
    ax[0].axhline(float(x_off[i]), color="k", ls=":", lw=0.6)
ax[0].set_title("Closed-loop: LQR on a Swi5 input steers the network back toward switch-off\n(dotted = switch-off targets)")
ax[0].set_xlabel("time"); ax[0].set_ylabel("expression"); ax[0].legend(fontsize=8)
ax[1].plot(ts, np.asarray(us)[:, 0]); ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_title("Control input  u(t) = -K(x - x_off)  (Swi5 actuation)"); ax[1].set_xlabel("time"); ax[1].set_ylabel("u")
plt.tight_layout(); plt.show()
print(f"||x - x_off||:  {float(jnp.linalg.norm(ys[0])):.3f}  ->  {float(jnp.linalg.norm(ys[-1])):.3e}")

## 5. Differentiable design: sensitivity of the control cost to the network

`jaxctrl`'s Riccati solver is autodiff-friendly, so we can ask how the LQR cost responds to a change in any regulatory strength — e.g. weakening the `Gal80 ⊣ Swi5` feedback edge in the identified model.

In [ ]:
i_swi5, i_gal80 = names.index("Swi5"), names.index("Gal80")

def cost_vs_feedback(scale):
    A_s = A_id.at[i_swi5, i_gal80].multiply(scale)            # rescale the Gal80 -> Swi5 entry
    em  = jnp.maximum(jnp.max(jnp.linalg.eigvals(A_s).real) + 0.2, 0.0)
    _, P_ = jaxctrl.lqr(A_s - em * jnp.eye(5), B, Q, R)
    return (x_on - x_off) @ P_ @ (x_on - x_off)

g = jax.grad(cost_vs_feedback)(1.0)
print(f"d(LQR cost)/d(Gal80->Swi5 feedback scale) at scale=1: {float(g):+.3f}")
scales = jnp.linspace(0.2, 2.0, 25)
costs  = jnp.array([cost_vs_feedback(s) for s in scales])
plt.figure(figsize=(6, 4)); plt.plot(scales, costs)
plt.xlabel("Gal80 ⊣ Swi5 feedback strength (× nominal)"); plt.ylabel("LQR cost-to-go")
plt.title("How the negative-feedback strength changes the cost of control"); plt.show()

---

**On the data.** This notebook drives the pipeline from a *stylized* Hill-ODE model with the IRMA wiring, so it is fully reproducible with no download. For the **experimental** IRMA time series (qPCR of CBF1/GAL4/SWI5/GAL80/ASH1 under galactose switch-on and switch-off, multiple replicates), see the supplementary data of **Cantone et al., "A yeast synthetic network for in vivo assessment of reverse-engineering and modeling approaches," *Cell* 137(1):172–181, 2009** (doi:10.1016/j.cell.2009.01.055). Drop those traces in as `X`/`dX` and the rest of the notebook runs unchanged. Other small real targets: the *E. coli* SOS network (~8 genes, Uri Alon lab), DREAM4 size-10, SERGIO/BEELINE simulated scRNA-seq.